# Secuenciación de trabajos: Job Shop

## Tipo de problema

Determinar el orden óptimo en el que una serie de trabajos deben pasar por distintas máquinas para minimizar el tiempo total de finalización (makespan) o los retrasos.

## Formas de resolverlo

Los nombres de la primera columna coinciden con la [tabla del README](../README.md#formas-de-resolverlo).

| Enfoque (README) | Aplicación en este problema | Uso en este ejercicio |
|---|---|---|
| Restricciones | Representar precedencias y ausencia de solapamientos; por ejemplo, con CP-SAT. | Alternativa |
| Entera o mixta | Modelar órdenes y tiempos mediante variables enteras y continuas. | Alternativa |
| Heurísticas y metaheurísticas | Buscar buenas agendas con búsqueda local, tabú o algoritmos genéticos. | Alternativa |
| Enumeración | Comparar todos los órdenes de las máquinas en esta instancia pequeña. | **Principal** |
| Grafos y redes | Descartar ciclos y calcular tiempos respetando precedencias. | **Complementario** |

**Enfoque utilizado:** Enumeración.

**Enfoque complementario:** Grafos y redes.

**Técnica concreta:** Enumeración de órdenes por máquina y cálculo de inicios más tempranos sobre el grafo de precedencias.

[Ejemplo de programación por restricciones: OR-Tools](https://developers.google.com/optimization/scheduling/job_shop).

## Ejercicio propuesto

Tres trabajos requieren dos máquinas en órdenes diferentes.  
J1: M1(3 h) -> M2(2 h); J2: M2(2 h) -> M1(1 h);  
J3: M1(2 h) -> M2(1 h). Una operación por máquina, sin interrupciones.  
Objetivo: minimizar el fin del último trabajo (makespan).  
Método exacto: enumerar órdenes por máquina y calcular inicios más tempranos  
sobre el grafo de precedencias. Los órdenes que crean ciclos se descartan.

## Método elegido para este ejercicio

**Enfoque utilizado:** Enumeración.

**Enfoque complementario:** Grafos y redes.

Enumeración exacta de los órdenes de cada máquina. Un grafo combina esos órdenes con las precedencias de los trabajos. Se descartan los ciclos y se calcula el comienzo más temprano de cada operación.

## Implementación

Ejecuta las celdas de código en orden. Solo se utiliza la biblioteca estándar de Python.

In [1]:
from itertools import permutations, product


def resolver():
    trabajos = (((0, 3), (1, 2)), ((1, 2), (0, 1)), ((0, 2), (1, 1)))
    ops = {(j, k): op for j, lista in enumerate(trabajos)
           for k, op in enumerate(lista)}
    grupos = [[n for n, (m, _) in ops.items() if m == maquina]
              for maquina in range(2)]
    mejor = float('inf'), None
    for ordenes in product(*(permutations(ns) for ns in grupos)):
        arcos = {((j, 0), (j, 1)) for j in range(3)}
        for orden in ordenes:
            arcos.update(zip(orden, orden[1:]))
        pendientes, fin = set(ops), {}
        while pendientes:
            listos = [n for n in pendientes
                      if all(a in fin for a, b in arcos if b == n)]
            if not listos:
                break
            for n in sorted(listos):
                inicio = max((fin[a] for a, b in arcos if b == n), default=0)
                fin[n] = inicio + ops[n][1]
                pendientes.remove(n)
        if not pendientes and max(fin.values()) < mejor[0]:
            mejor = max(fin.values()), fin
    total, fin = mejor
    assert fin is not None
    assert all(fin[j, 0] <= fin[j, 1] - trabajos[j][1][1] for j in range(3))
    agenda = [(j + 1, k + 1, ops[j, k][0] + 1,
               fin[j, k] - ops[j, k][1], fin[j, k]) for j, k in fin]
    for maquina in (1, 2):
        intervalos = sorted((a, b) for _, _, m, a, b in agenda if m == maquina)
        assert all(b <= c for (_, b), (c, _) in zip(intervalos, intervalos[1:]))
    return total, sorted(agenda)

## Resolución y resultados

In [2]:
total, agenda = resolver()
for j, op, m, inicio, fin in agenda:
    print(f'J{j}, operación {op}, M{m}: {inicio}-{fin} h')
print(f'Makespan mínimo: {total} h')

J1, operación 1, M1: 0-3 h
J1, operación 2, M2: 3-5 h
J2, operación 1, M2: 0-2 h
J2, operación 2, M1: 5-6 h
J3, operación 1, M1: 3-5 h
J3, operación 2, M2: 5-6 h
Makespan mínimo: 6 h


## Interpretación

El último trabajo termina a las **6 h**. Es óptimo: M1 necesita 3+1+2=6 h de procesamiento, por lo que ninguna agenda puede terminar antes. Los intervalos impresos permiten comprobar que no hay solapamientos.

**Alcance:** los datos son sintéticos. Las garantías y supuestos se refieren a esta instancia; al aumentar su tamaño, la enumeración puede resultar costosa.